# SmartBank AI — Exploratory Data Analysis

Run `python -m src.data.generate` from the project root before executing these cells. The dataset is synthetic and reproducible.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.config import DEFAULT_DATA_DIR

transactions = pd.read_csv(DEFAULT_DATA_DIR / 'transactions.csv', parse_dates=['timestamp'])
customers = pd.read_csv(DEFAULT_DATA_DIR / 'customers.csv')
transactions.head()

## Dataset checks

Review row counts, missing values and duplicate keys before interpreting charts.

In [ ]:
print(f'{len(transactions):,} transactions | {transactions.customer_id.nunique():,} customers')
display(transactions.isna().sum().sort_values(ascending=False).rename('missing values').to_frame())
print('Duplicate transaction IDs:', transactions.transaction_id.duplicated().sum())

In [ ]:
fraud_rate = transactions.fraud.mean() * 100
fig = px.histogram(transactions, x='fraud', color='fraud', title=f'Fraud label distribution ({fraud_rate:.2f}% positive)', category_orders={'fraud': [0, 1]})
fig.show()

In [ ]:
fig = px.histogram(transactions, x='amount', color='fraud', nbins=80, barmode='overlay', opacity=.65,
                   title='Transaction amount distribution by fraud label', log_y=True)
fig.show()

In [ ]:
def fraud_rate_by(column):
    summary = transactions.groupby(column, dropna=False).fraud.agg(['mean', 'count']).reset_index()
    summary['fraud_percent'] = summary['mean'] * 100
    return px.bar(summary, x=column, y='fraud_percent', color='count', title=f'Fraud rate by {column}',
                  labels={'fraud_percent': 'Fraud rate (%)', 'count': 'Transactions'})

for column in ['location', 'transaction_type', 'device_type']:
    fraud_rate_by(column).show()

In [ ]:
hourly = transactions.groupby('hour').fraud.mean().mul(100).rename('fraud_percent').reset_index()
px.line(hourly, x='hour', y='fraud_percent', markers=True, title='Fraud rate by hour of day').show()

In [ ]:
px.histogram(customers, x='credit_score', color='churn', nbins=35, barmode='overlay', opacity=.65,
             title='Customer credit score distribution by churn label').show()
px.histogram(customers, x='risk_score', color='risk_category', title='Customer risk score distribution').show()

In [ ]:
numeric = transactions.select_dtypes(include='number').drop(columns=['fraud'], errors='ignore')
corr = numeric.corr()
px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
          title='Numeric transaction feature correlations').show()

## Reading the results

Fraud is intentionally uncommon. Compare rates and sample counts together: a high rate in a small category may be noisy. These charts describe synthetic data and should not be treated as real-world banking findings. Correlations are descriptive and do not establish causality.